# Fraud Detection Model Training

This notebook trains classification models to detect fraudulent transactions.

The goal is to compare a simple baseline model with real machine learning models and evaluate which model performs better for fraud detection.

In [13]:
# Import libraries for data analysis
import pandas as pd
import numpy as np

# Import libraries for preprocessing and splitting data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Import machine learning models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Import evaluation metrics
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Import libraries for visualization
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Load Cleaned Dataset

In [14]:
# Define the path to the cleaned dataset
CLEANED_DATA = "../data/processed/creditcard_cleaned.csv"

# Load the cleaned dataset
df = pd.read_csv(CLEANED_DATA)

# Display dataset shape
print("Dataset shape:")
print(df.shape)

# Display first rows
display(df.head())

Dataset shape:
(283726, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


## 2. Prepare Features and Target

Separate the input features from the target variable.

`X` contains transaction features.  
`y` contains the fraud label.

In [15]:
# Create feature matrix by removing the target column
X = df.drop("Class", axis=1)

# Create target variable
y = df["Class"]

# Display feature and target shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (283726, 30)
y shape: (283726,)


**Result:** The feature matrix `X` contains **283,726 rows** and **30 input features**.  
The target variable `y` contains **283,726 fraud labels**.

## 3. Split datasets in Train/Test

Split the dataset into training and test sets.

Stratified split is used because fraud transactions are rare.

In [16]:
# Split data into train and test sets while preserving class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Display train and test shapes
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

# Check class distribution in train set
print("\nTrain class distribution:")
display(y_train.value_counts(normalize=True) * 100)

# Check class distribution in test set
print("\nTest class distribution:")
display(y_test.value_counts(normalize=True) * 100)

X_train shape: (226980, 30)
X_test shape: (56746, 30)

Train class distribution:


Class
0    99.833466
1     0.166534
Name: proportion, dtype: float64


Test class distribution:


Class
0    99.832587
1     0.167413
Name: proportion, dtype: float64

**Result:** The data was split into training and test sets.  
`X_train` contains **226,980 rows** and `X_test` contains **56,746 rows**.  
The fraud ratio was preserved in both sets using stratified splitting.

## 4. Scaling for Logistic Regression

Scale `Time` and `Amount` for Logistic Regression.

The anonymized features `V1`–`V28` are already transformed.

In [17]:
# Create copies for scaled data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Create scaler object
scaler = StandardScaler()

# Scale Time and Amount using training data
X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(
    X_train[["Time", "Amount"]]
)

# Apply the same scaling to test data
X_test_scaled[["Time", "Amount"]] = scaler.transform(
    X_test[["Time", "Amount"]]
)

# Display scaled training data preview
display(X_train_scaled.head())

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
225399,1.045499,2.238954,-1.724499,-2.151484,-2.577803,0.993668,3.565492,-1.785957,0.860122,-1.264003,...,-0.323810,-0.149574,-0.049333,0.278442,0.684735,-0.219028,-0.159167,0.037920,-0.049932,-0.229434
133746,-0.298690,-1.315062,1.630783,0.597001,-0.038359,-0.404580,-0.965712,0.212249,0.735381,-1.267926,...,-0.067580,-0.238898,-0.946773,0.323904,0.515632,-0.713000,-0.266503,-0.017794,0.051058,-0.331197
185792,0.678397,1.908801,0.021184,-2.087997,0.129310,1.161468,0.605244,-0.022371,0.180296,0.283819,...,-0.210474,0.293609,1.095842,-0.044874,-1.689517,0.106098,0.007758,0.045164,-0.053068,-0.298809
148925,-0.074929,1.811257,0.316556,0.316751,3.880231,0.048454,1.020163,-0.734868,0.233651,0.681423,...,-0.228032,0.138869,0.700422,0.174064,0.702997,-0.212523,-0.010018,-0.017740,-0.038006,-0.289247
18398,-1.376728,1.358817,-1.120881,0.550266,-1.547659,-1.194950,0.275448,-1.201843,0.212889,-2.094285,...,-0.361686,-0.340972,-0.636442,0.252758,-0.344160,-0.064282,-0.439622,0.062524,0.013095,-0.261985


**Result:** The `Time` and `Amount` features were scaled in the training and test sets.  
The scaled datasets keep the same number of rows and columns as the original train/test data.

## 5. DummyClassifier

Train a simple baseline model.

This model helps compare real ML models against a naive strategy.

In [18]:
# Create baseline model
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

# Train baseline model
dummy_model.fit(X_train_scaled, y_train)

# Predict classes
y_pred_dummy = dummy_model.predict(X_test_scaled)

# Predict fraud probabilities
y_proba_dummy = dummy_model.predict_proba(X_test_scaled)[:, 1]

# Display classification report
print(classification_report(y_test, y_pred_dummy, zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.00      0.00      0.00        95

    accuracy                           1.00     56746
   macro avg       0.50      0.50      0.50     56746
weighted avg       1.00      1.00      1.00     56746



## Result — DummyClassifier predicted all transactions as normal.

For this dataset, this gives high accuracy because normal transactions strongly dominate the data.

However, the model detected **0 fraud transactions**.

Fraud precision, recall, and F1-score are all **0.00**.

This confirms that accuracy alone is misleading for this fraud detection project.


## 6. Logistic Regression

Train Logistic Regression as the first real classification model.

It is simple, interpretable, and suitable for binary fraud classification.

In [19]:

# Create Logistic Regression model
logreg_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

# Train Logistic Regression model
logreg_model.fit(X_train_scaled, y_train)

# Predict classes
y_pred_logreg = logreg_model.predict(X_test_scaled)

# Predict fraud probabilities
y_proba_logreg = logreg_model.predict_proba(X_test_scaled)[:, 1]

# Display classification report
print(classification_report(y_test, y_pred_logreg, zero_division=0))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56651
           1       0.06      0.87      0.11        95

    accuracy                           0.98     56746
   macro avg       0.53      0.92      0.55     56746
weighted avg       1.00      0.98      0.99     56746



## Result — Logistic Regression

Logistic Regression detected most fraud cases, with fraud recall of **0.87**.

This means the model found about **87% of real fraudulent transactions** in the test set.

However, fraud precision is only **0.06**, which means many normal transactions were also flagged as fraud.

For this project, Logistic Regression is useful because it catches many fraud cases, but it would create a high manual review workload.

This model is good for understanding the fraud detection trade-off between recall and false positives.


## 7. Random Forest

Train Random Forest as a stronger nonlinear model.

This model can capture more complex fraud patterns.

In [20]:
# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train Random Forest model
rf_model.fit(X_train, y_train)

# Predict classes
y_pred_rf = rf_model.predict(X_test)

# Predict fraud probabilities
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# Display classification report
print(classification_report(y_test, y_pred_rf, zero_division=0))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.97      0.71      0.82        95

    accuracy                           1.00     56746
   macro avg       0.99      0.85      0.91     56746
weighted avg       1.00      1.00      1.00     56746



## Result — Random Forest

Random Forest achieved fraud precision of **0.97** and fraud recall of **0.71**.

This means that when the model flags a transaction as fraud, it is usually correct.

However, it detects fewer fraud cases than Logistic Regression.

The F1-score for fraud is **0.82**, which is much stronger than Logistic Regression.

For this project, Random Forest is the strongest model because it gives a better balance between fraud detection and false alarms.


## 8. Model Evaluation

Create one evaluation function to measure all models with the same metrics.

For fraud detection, precision, recall, F1-score, ROC-AUC, and PR-AUC are more useful than accuracy alone.

In [21]:
# This function evaluates a classification model using fraud detection metrics
def evaluate_model(model_name, y_true, y_pred, y_proba):
    results = {
        "model": model_name,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba)
    }

    return results


# Evaluate DummyClassifier
dummy_results = evaluate_model(
    "DummyClassifier",
    y_test,
    y_pred_dummy,
    y_proba_dummy
)

# Evaluate Logistic Regression
logreg_results = evaluate_model(
    "Logistic Regression",
    y_test,
    y_pred_logreg,
    y_proba_logreg
)

# Evaluate Random Forest
rf_results = evaluate_model(
    "Random Forest",
    y_test,
    y_pred_rf,
    y_proba_rf
)

# Display individual results
display(dummy_results)
display(logreg_results)
display(rf_results)

{'model': 'DummyClassifier',
 'precision': 0.0,
 'recall': 0.0,
 'f1_score': 0.0,
 'roc_auc': np.float64(0.5),
 'pr_auc': np.float64(0.0016741268107003137)}

{'model': 'Logistic Regression',
 'precision': 0.05623306233062331,
 'recall': 0.8736842105263158,
 'f1_score': 0.10566518141311267,
 'roc_auc': np.float64(0.9658392242808925),
 'pr_auc': np.float64(0.6719353631354262)}

{'model': 'Random Forest',
 'precision': 0.9710144927536232,
 'recall': 0.7052631578947368,
 'f1_score': 0.8170731707317073,
 'roc_auc': np.float64(0.9246078250116827),
 'pr_auc': np.float64(0.7959572053024366)}

## Result — Model Evaluation

All models were evaluated using the same fraud detection metrics: precision, recall, F1-score, ROC-AUC, and PR-AUC.

DummyClassifier shows why accuracy is misleading in this dataset.

Logistic Regression gives high fraud recall but low precision.

Random Forest gives much higher precision and stronger F1-score for fraud detection.

This evaluation shows that fraud detection should be measured with precision, recall, F1-score, PR-AUC, and ROC-AUC, not accuracy alone.

## 9. Model Comparison

Compare all trained models in one table.

This helps select the most suitable model for fraud detection.

In [22]:
# Combine all model results into one list
model_results = [
    dummy_results,
    logreg_results,
    rf_results
]

# Convert results into a DataFrame
model_results_df = pd.DataFrame(model_results)

# Display model comparison table
display(model_results_df)

,model,precision,recall,f1_score,roc_auc,pr_auc
0,DummyClassifier,0.000000,0.000000,0.000000,0.500000,0.001674
1,Logistic Regression,0.056233,0.873684,0.105665,0.965839,0.671935
2,Random Forest,0.971014,0.705263,0.817073,0.924608,0.795957


## Result — Model Comparison

The comparison shows that DummyClassifier is not useful for fraud detection because it detects no fraud cases.

Logistic Regression has higher recall, so it catches more fraud transactions, but it also creates many false positives.

Random Forest has much higher precision and the best PR-AUC score of **0.7960**.

For this project, Random Forest is the best model for the next step because it provides the strongest overall balance between precision, recall, F1-score, and PR-AUC.

## 10. Model Training Summary

Three models were trained and compared:

- **DummyClassifier** was used as a baseline model.
- **Logistic Regression** was used as a simple and interpretable classification model.
- **Random Forest** was used as a stronger nonlinear model.

The dataset is highly imbalanced, so accuracy is not the main evaluation metric.

The most important metrics are:

- **Recall** — shows how many real fraud transactions the model detects.
- **Precision** — shows how many flagged transactions are actually fraud.
- **F1-score** — balances precision and recall.
- **PR-AUC** — useful for imbalanced fraud detection.
- **ROC-AUC** — shows general model separation ability.

**Result:** The model training workflow produced three comparable models: a naive baseline, an interpretable linear model, and a stronger nonlinear model.  
The best model can now be connected to threshold analysis using predicted fraud probabilities.


## Save Random Forest Predictions for Threshold Analysis

Save Random Forest fraud probabilities and test data for the next notebook.

These probabilities will be used to test different fraud decision thresholds.

In [24]:
# Import Path to create folders if needed
from pathlib import Path

# Define output path for threshold analysis input
PREDICTIONS_OUTPUT_PATH = "../reports/random_forest_predictions.csv"

# Create a copy of the test data
threshold_input_df = X_test.copy()

# Add true fraud labels
threshold_input_df["y_true"] = y_test.values

# Add Random Forest predicted fraud probabilities
threshold_input_df["fraud_probability"] = y_proba_rf

# Add Random Forest default predictions at threshold 0.50
threshold_input_df["prediction_050"] = y_pred_rf

# Create reports folder if it does not exist
Path("../reports").mkdir(parents=True, exist_ok=True)

# Save prediction data for threshold analysis
threshold_input_df.to_csv(PREDICTIONS_OUTPUT_PATH, index=False)

# Confirm saved file
print(f"Random Forest predictions saved to: {PREDICTIONS_OUTPUT_PATH}")

Random Forest predictions saved to: ../reports/random_forest_predictions.csv
